# UNIT07 YOLO 客製化模型訓練與 ONNX 匯出
學生版 v4｜2026-10-16

先開啟「執行階段 → 變更執行階段類型 → GPU」。依序執行 01–10，不要按「全部執行」。每格成功才執行下一格。

準備：照片 ZIP、YOLO 標註 ZIP、標註時的類別順序。本筆記本支援單類與多類物件偵測；不適用分類或分割資料。

檔案會上傳至 Google Colab。只使用教師同意的課堂素材。Colab 免費 GPU 不保證可用；不要在訓練時關閉分頁。

沿用課堂已使用的 Ultralytics 8.4.177、YOLO26n、640、50 epochs。完成後必須下載成果；筆記本存檔不等於模型已備份。

## 01 安裝工具與確認 GPU
按左側執行鈕。第一次需等待安裝。成功會顯示版本、GPU 名稱與「環境準備完成」。若要求重新啟動工作階段，重新啟動後再執行本格。

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "ultralytics==8.4.177", "onnx", "onnxruntime", "ipywidgets"])
import torch, ultralytics
print("Ultralytics:", ultralytics.__version__)
assert ultralytics.__version__ == "8.4.177", "請重新啟動工作階段，再執行 01。"
assert torch.cuda.is_available(), "沒有 GPU。請改成 GPU 執行階段，再從 01 開始。"
print("GPU:", torch.cuda.get_device_name(0))
print("環境準備完成")

## 02 上傳照片 ZIP
只選一個照片 ZIP，包含全部類別的 JPG、JPEG 或 PNG。不要在這格選標註 ZIP。ZIP 可有子資料夾，但每張照片的主檔名必須唯一。

In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile
uploaded_images = files.upload()
assert len(uploaded_images) == 1, "一次只選一個照片 ZIP。請重跑 02。"
image_zip_name, image_zip_bytes = next(iter(uploaded_images.items()))
assert zipfile.is_zipfile(io.BytesIO(image_zip_bytes)), "這不是有效 ZIP。"
print("照片 ZIP 已上傳：", image_zip_name)

## 03 上傳 YOLO 標註 ZIP
選擇 Make Sense AI 匯出的 YOLO 標註 ZIP。應含與照片同名的 .txt，不是把畫好框的照片當標註。

In [ ]:
uploaded_labels = files.upload()
assert len(uploaded_labels) == 1, "一次只選一個標註 ZIP。請重跑 03。"
label_zip_name, label_zip_bytes = next(iter(uploaded_labels.items()))
assert zipfile.is_zipfile(io.BytesIO(label_zip_bytes)), "這不是有效 ZIP。"
print("標註 ZIP 已上傳：", label_zip_name)

## 04 自動檢查 ID 與類別對應
直接執行，不必修改程式碼。程式會將同名照片與標註配對，檢查整份資料的「檔名類別」與「TXT 類別 ID」。

若照片採用 `類別名稱_流水號.jpg`，而且每張照片的標註只有一種類別 ID，程式會統計全部照片。對應一致時，直接列出正確的檔名對照並填入表單；即使 classes.txt 的順序相反，也會顯示警告並修正本次訓練的名稱順序，不交換標註 ID。

例如這次資料會顯示：`ID 0 = papers（30 張照片一致）`、`ID 1 = scissors（30 張照片一致）`。其他資料集依實際結果判定，不固定交換 0 與 1。

**這是檔名與標註的一致性檢查，不是 AI 看圖辨識。** 如果照片檔名本身錯誤，仍需要人工修正。多類物件同時出現在同一張照片，或檔名不含類別時，不會猜測名稱，改用名稱檔與照片預覽核對。同一 ID 對應多個檔名類別時會停止並指出衝突。

下方每個 ID 都有照片與紅框，並可瀏覽全部標註照片。核對後勾選各類並按「確認全部類別並繼續」，再執行 05。若同一 ID 混入不同物件，請先回標註工具修正。


In [ ]:
# 直接執行本格，不必修改程式。
import io, zipfile, math, hashlib, yaml
import ipywidgets as widgets
from IPython.display import display, clear_output
CLASS_NAMES = []
CLASS_SIGNATURE = None
REVIEW_SIGNATURE = None

def dataset_signature(images, labels):
    return (hashlib.sha256(images).hexdigest(), hashlib.sha256(labels).hexdigest())

def normalize_names(value):
    if isinstance(value, dict):
        converted = {}
        for key, name in value.items():
            assert str(key).isdigit(), "名稱檔的 ID 必須是從 0 開始的整數。"
            index = int(key)
            assert index not in converted, "名稱檔有重複的 ID。"
            converted[index] = name
        assert sorted(converted) == list(range(len(converted))), "名稱檔 ID 必須連續且從 0 開始。"
        value = [converted[i] for i in range(len(converted))]
    assert isinstance(value, list) and value, "名稱檔沒有有效的類別清單。"
    assert all(isinstance(n, str) and n.strip() for n in value), "類別名稱不可空白。"
    names = [n.strip() for n in value]
    assert len(names) == len(set(names)), "類別名稱不能重複。"
    assert all(not any(c in n for c in "\n\r,;") for n in names), "名稱不可含換行、逗號或分號。"
    return names

def discover_classes(image_blob, label_blob):
    candidates, observed = [], set()
    for source, blob in (("照片 ZIP", image_blob), ("標註 ZIP", label_blob)):
        with zipfile.ZipFile(io.BytesIO(blob)) as archive:
            for info in archive.infolist():
                path = Path(info.filename)
                if info.is_dir() or "__MACOSX" in path.parts or path.name.startswith("."):
                    continue
                filename = path.name.lower()
                if filename in ("classes.txt", "obj.names"):
                    value = archive.read(info).decode("utf-8-sig").splitlines()
                    candidates.append((source+" / "+info.filename, normalize_names(value)))
                elif path.suffix.lower() in (".yaml", ".yml"):
                    value = yaml.safe_load(archive.read(info).decode("utf-8-sig"))
                    if isinstance(value, dict) and "names" in value:
                        candidates.append((source+" / "+info.filename, normalize_names(value["names"])))
                elif source == "標註 ZIP" and path.suffix.lower() == ".txt":
                    for line in archive.read(info).decode("utf-8-sig").splitlines():
                        if not line.strip():
                            continue
                        parts = line.split()
                        assert len(parts) == 5, f"{info.filename} 不是 YOLO 偵測五欄格式。"
                        cid = float(parts[0])
                        assert math.isfinite(cid) and cid.is_integer() and 0 <= cid < 1000, "標註 ID 無效，請回標註工具檢查。"
                        observed.add(int(cid))
    assert observed, "找不到任何有效標註，請確認 03 的 ZIP。"
    if candidates:
        names = candidates[0][1]
        assert all(v == names for _, v in candidates), "ZIP 內的類別名稱檔互相衝突。請保留正確版本，再重新上傳。"
        assert max(observed) < len(names), "標註 ID 超出名稱檔範圍，請確認照片與標註來自同一專案。"
        return names, [src for src, _ in candidates], observed
    return None, [], observed


from PIL import Image, ImageDraw, ImageFont
import random

def read_label_rows(text):
    rows = []
    for line in text.splitlines():
        if not line.strip(): continue
        parts = line.split()
        assert len(parts) == 5, "標註必須是 YOLO 物件偵測的五欄格式。"
        cid, x, y, w, h = map(float, parts)
        assert all(math.isfinite(v) for v in (cid,x,y,w,h)), "標註含無效數字。"
        assert cid.is_integer() and cid >= 0, "標註 ID 必須是非負整數。"
        assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1, "框線座標不合法。"
        assert x-w/2 >= -.001 and y-h/2 >= -.001 and x+w/2 <= 1.001 and y+h/2 <= 1.001, "標註框超出影像。"
        rows.append((int(cid),x,y,w,h))
    return rows

def preview_index(images, labels):
    photos, annotations = {}, {}
    for blob, target, suffixes in ((images,photos,{".png",".jpg",".jpeg"}),
                                  (labels,annotations,{".txt"})):
        with zipfile.ZipFile(io.BytesIO(blob)) as archive:
            for info in archive.infolist():
                path=Path(info.filename)
                if info.is_dir() or "__MACOSX" in path.parts or path.name.startswith("."): continue
                if path.name.lower()=="classes.txt" or path.suffix.lower() not in suffixes: continue
                assert path.stem not in target, f"重複主檔名：{path.stem}，請修正資料。"
                target[path.stem]=(path.name,archive.read(info))
    assert photos, "照片 ZIP 沒有可用的照片。"
    assert set(photos)==set(annotations), "照片與標註檔名未完全配對，請確認兩個 ZIP 是同一批資料。"
    indexed={}
    for stem, (_, blob) in annotations.items():
        for row in read_label_rows(blob.decode("utf-8-sig")):
            indexed.setdefault(row[0],{}).setdefault(stem,[]).append(row)
    return photos,indexed

def preview_png(blob, rows):
    with Image.open(io.BytesIO(blob)) as source:
        im=source.convert("RGB")
    im.thumbnail((480,360))
    draw=ImageDraw.Draw(im)
    try: font=ImageFont.truetype("DejaVuSans.ttf",16)
    except OSError: font=ImageFont.load_default()
    w,h=im.size
    for cid,x,y,bw,bh in rows:
        box=(max(0,(x-bw/2)*w),max(0,(y-bh/2)*h),min(w,(x+bw/2)*w),min(h,(y+bh/2)*h))
        draw.rectangle(box,outline="#ff3030",width=3)
        text=f"ID {cid}";left,top=box[:2]
        tw=draw.textbbox((0,0),text,font=font)[2]+8
        top=max(0,top-22)
        draw.rectangle((left,top,left+tw,top+22),fill="#b00000")
        draw.text((left+4,top+1),text,font=font,fill="white")
    stream=io.BytesIO();im.save(stream,format="PNG");return stream.getvalue()


def infer_filename_mapping(photos, groups, declared=None):
    # Only use the course's class_name_001 naming scheme, and single-class photos.
    # This checks filename/annotation consistency; it is not visual recognition.
    import re
    per_photo={stem:set() for stem in photos}
    for cid, members in groups.items():
        for stem in members: per_photo[stem].add(cid)
    evidence={}
    unsupported=[]
    for stem in sorted(photos):
        ids=per_photo[stem]
        if not ids: continue  # Background photos do not define class names.
        if len(ids)!=1:
            return None,{},"有照片包含多個類別 ID，無法用整張照片的檔名替各個框命名。"
        match=re.fullmatch(r"(.+)_([0-9]+)",stem)
        if not match:
            unsupported.append(stem);continue
        name=match.group(1)
        if declared is not None and name not in declared:
            unsupported.append(stem);continue
        cid=next(iter(ids))
        evidence.setdefault(cid,{}).setdefault(name,[]).append(stem)
    if unsupported:
        return None,{},"部分檔名不是已知的 類別名稱_流水號 格式，例如："+", ".join(unsupported[:3])
    if not evidence:
        return None,{},"沒有可用的檔名與標註配對。"
    conflicts={cid:list(names) for cid,names in evidence.items() if len(names)!=1}
    if conflicts:
        raise ValueError(f"自動檢查發現衝突：同一 ID 對應不同檔名類別 {conflicts}。請先檢查照片或標註，不可直接訓練。")
    ids=sorted(evidence)
    if ids!=list(range(len(ids))):
        raise ValueError("標註 ID 不連續，請先修正資料。")
    names=[next(iter(evidence[cid])) for cid in ids]
    if len(names)!=len(set(names)):
        raise ValueError("同一檔名類別被標成不同 ID，請先修正標註。")
    if declared is not None and set(names)!=set(declared):
        return None,{},"檔名類別與名稱檔不完整對應，需人工核對。"
    counts={cid:len(next(iter(evidence[cid].values()))) for cid in ids}
    return names,counts,""

if globals().get("class_form") is not None:
    class_form.close()
auto_names, sources, observed_ids = discover_classes(image_zip_bytes,label_zip_bytes)
photos_by_name, examples_by_id = preview_index(image_zip_bytes,label_zip_bytes)
original_names = list(auto_names) if auto_names else None
inferred_names, mapping_counts, mapping_reason = infer_filename_mapping(
    photos_by_name, examples_by_id, original_names)
if inferred_names:
    auto_names = inferred_names
    print("自動檢查完成：以下對應來自整份資料的『照片檔名＋標註 ID』交叉比對。")
    for cid, name in enumerate(auto_names):
        print(f"ID {cid} = {name}（{mapping_counts[cid]} 張照片一致）")
    if original_names is not None and original_names != auto_names:
        print("注意：ZIP 的類別名稱順序與實際標註相反或不同，已自動修正本次訓練的名稱對照。")
        print("原名稱檔：", dict(enumerate(original_names)))
        print("修正後：", dict(enumerate(auto_names)))
    print("標註 TXT 的 ID 保持不變。這是檔名一致性檢查，請用下方照片核對物件內容。")
else:
    print("無法從檔名自動確認：", mapping_reason)
    print("下方名稱僅為名稱檔提供的候選值，請看照片確認。")

this_signature=dataset_signature(image_zip_bytes,label_zip_bytes)
count=len(auto_names) if auto_names else max(observed_ids)+1
assert set(examples_by_id)==set(range(count)), "有類別 ID 沒有標註範例，請先補齊或修正類別 ID。"
print("已帶入類別對應；請看照片核對，必要時可直接修正欄位。" if auto_names else "請看每個 ID 的紅框物件，再填名稱，不用改程式。")
print("每組只畫出該 ID 的框；同一張照片可能包含其他物件。若同組混入不同物件，請先修正標註。")
print("範例不代表全部資料都正確；可切換照片瀏覽該 ID 的所有標註照片。")
fields=[];checks=[];sections=[];viewers=[]

def make_id_viewer(cid):
    stems=sorted(examples_by_id[cid])
    sample=list(stems);random.Random(42+cid).shuffle(sample)
    tiles=[]
    for stem in sample[:3]:
        pic=widgets.Image(value=preview_png(photos_by_name[stem][1], examples_by_id[cid][stem]),format="png",layout=widgets.Layout(width="240px"))
        tiles.append(widgets.VBox([pic,widgets.Label(value=photos_by_name[stem][0])]))
    picker=widgets.Dropdown(options=[(photos_by_name[k][0],k) for k in stems],description="查看照片",layout=widgets.Layout(width="95%"))
    full=widgets.Image(format="png",layout=widgets.Layout(max_width="480px"))
    def show(change=None):
        stem=picker.value
        full.value=preview_png(photos_by_name[stem][1],examples_by_id[cid][stem])
    picker.observe(show,names="value");show()
    viewers.append((picker,full))
    return [widgets.Label(value=f"ID {cid}｜共 {len(stems)} 張照片，先看下方紅框內是什麼物件"),
            widgets.Box(tiles,layout=widgets.Layout(display="flex",flex_flow="row wrap")),picker,full]

for cid in range(count):
    field=widgets.Text(value=auto_names[cid] if auto_names else "",description=f"ID {cid} 名稱",placeholder="看上方紅框物件，填入名稱",layout=widgets.Layout(width="95%"))
    check=widgets.Checkbox(value=False,description=f"我已核對 ID {cid} 的物件與名稱相符",indent=False,layout=widgets.Layout(width="95%"))
    fields.append(field);checks.append(check)
    sections.append(widgets.VBox(make_id_viewer(cid)+[field,check]))
confirm=widgets.Button(description="確認全部類別並繼續",button_style="success",layout=widgets.Layout(width="240px"))
feedback=widgets.Output()

def invalidate_names(change):
    global CLASS_NAMES,CLASS_SIGNATURE,REVIEW_SIGNATURE
    CLASS_NAMES,CLASS_SIGNATURE,REVIEW_SIGNATURE=[],None,None
    if change.get("owner") in fields:
        checks[fields.index(change["owner"])].value=False
    with feedback:
        clear_output();print("設定已變更，請重新核對並確認。")
for field in fields: field.observe(invalidate_names,names="value")
for check in checks: check.observe(invalidate_names,names="value")

def confirm_names(button):
    global CLASS_NAMES,CLASS_SIGNATURE,REVIEW_SIGNATURE
    with feedback:
        clear_output()
        try:
            assert this_signature==dataset_signature(image_zip_bytes,label_zip_bytes), "ZIP 已變更，請重跑 04。"
            names=normalize_names([field.value for field in fields])
            assert all(check.value for check in checks), "請逐類看照片並勾選核對，再按確認。"
            CLASS_NAMES,CLASS_SIGNATURE,REVIEW_SIGNATURE=names,this_signature,None
            for i,name in enumerate(names): print(f"ID {i} = {name}")
            print("類別確認完成。請重新執行 05，再到 06 核對。原本 TXT 的 ID 不會被交換。")
        except Exception as error:
            CLASS_NAMES,CLASS_SIGNATURE,REVIEW_SIGNATURE=[],None,None
            print("尚未完成：",str(error))
confirm.on_click(confirm_names)
class_form=widgets.VBox(sections+[confirm,feedback]);display(class_form)

## 05 檢查資料與建立訓練集
不需改程式。會配對照片與標註、檢查框線數值、分成約 80% 訓練與 20% 驗證，並產生 data.yaml。

本 SOP 要求每張照片都有同名 TXT；已確認沒有目標物的照片可用空白 TXT。缺 TXT 會停止，避免把漏標照片誤當背景。

每類至少要出現在 2 張不同照片，才能分到兩邊；這只是檢查下限，不代表足夠訓練。連拍的近似照片應先篩選，以免驗證分數虛高。

In [ ]:
REVIEW_SIGNATURE = None
DATA_CLASS_NAMES = None
DATA_SOURCE_SIGNATURE = None
assert CLASS_NAMES and CLASS_SIGNATURE == dataset_signature(image_zip_bytes, label_zip_bytes), "請先執行 04 並在表單按確認；若重新上傳 ZIP，需再次執行 04。"
import random, math, shutil, hashlib, yaml
from PIL import Image

def zip_items(blob, extensions):
    found = {}
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        for info in archive.infolist():
            p = Path(info.filename)
            if info.is_dir() or "__MACOSX" in p.parts or p.name.startswith("."):
                continue
            if p.suffix.lower() not in extensions:
                continue
            if p.stem in found:
                raise ValueError(f"重複主檔名：{p.stem}，請先重新命名並同步標註。")
            found[p.stem] = (p.suffix.lower(), archive.read(info))
    return found

def prepare_dataset(image_blob, label_blob, names, root):
    photos = zip_items(image_blob, {".jpg", ".jpeg", ".png"})
    labels = zip_items(label_blob, {".txt"})
    labels = {k:v for k,v in labels.items() if k.lower() != "classes"}
    assert len(photos) >= 2, "照片不足，請確認 02 上傳的是照片 ZIP。"
    missing = sorted(set(photos)-set(labels))
    assert not missing, f"照片缺少同名 TXT：{missing[:8]}。請補標後重新上傳。"
    extra = sorted(set(labels)-set(photos))
    assert not extra, f"標註找不到照片：{extra[:8]}。請確認兩個 ZIP 是同一批資料。"
    ids, hashes = {}, set()
    for stem, (_, blob) in photos.items():
        with Image.open(io.BytesIO(blob)) as im:
            im.verify()
        digest = hashlib.sha256(blob).hexdigest()
        assert digest not in hashes, f"重複照片：{stem}。請移除重複檔與對應標註。"
        hashes.add(digest)
        ids[stem] = set()
        text = labels[stem][1].decode("utf-8-sig")
        for line_no, line in enumerate(text.splitlines(), 1):
            if not line.strip():
                continue
            parts = line.split()
            assert len(parts) == 5, f"{stem} 第 {line_no} 行不是 YOLO 偵測的 5 欄格式。"
            cid, x, y, w, h = map(float, parts)
            assert all(math.isfinite(v) for v in (cid,x,y,w,h)), f"{stem} 有無效數值。"
            assert cid.is_integer() and 0 <= cid < len(names), f"{stem} 類別 ID 超出 04 範圍。"
            assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1, f"{stem} 框線不是 0 到 1 的正規化座標。"
            assert x-w/2 >= -0.001 and y-h/2 >= -0.001 and x+w/2 <= 1.001 and y+h/2 <= 1.001, f"{stem} 框線超出影像，請回標註工具檢查。"
            ids[stem].add(int(cid))
    all_ids = set(range(len(names)))
    for cid, name in enumerate(names):
        n = sum(cid in v for v in ids.values())
        assert n >= 2, f"{name} 只出現在 {n} 張照片，請增加不同照片並標註。"
    keys = sorted(photos)
    n_val = max(1, round(len(keys)*0.2))
    rng = random.Random(42)
    for attempt in range(2000):
        rng.shuffle(keys)
        val, train = keys[:n_val], keys[n_val:]
        if set.union(set(), *(ids[k] for k in val)) == all_ids and set.union(set(), *(ids[k] for k in train)) == all_ids:
            break
    else:
        raise ValueError("無法讓所有類別都分到訓練與驗證集。請增加各類照片，或請老師協助分組。")
    root = Path(root)
    if root.exists():
        shutil.rmtree(root)
    for split, subset in (("train", train), ("val", val)):
        (root/"images"/split).mkdir(parents=True)
        (root/"labels"/split).mkdir(parents=True)
        for stem in subset:
            suffix, blob = photos[stem]
            (root/"images"/split/(stem+suffix)).write_bytes(blob)
            (root/"labels"/split/(stem+".txt")).write_bytes(labels[stem][1])
    config = {"path":str(root), "train":"images/train", "val":"images/val",
              "names":dict(enumerate(names))}
    (root/"data.yaml").write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8")
    (root/"classes.txt").write_text("\n".join(names), encoding="utf-8")
    print(f"資料檢查通過：{len(photos)} 張；train {len(train)}；val {len(val)}")
    for cid, name in enumerate(names):
        print(name, "訓練照片", sum(cid in ids[k] for k in train), "驗證照片", sum(cid in ids[k] for k in val))
    return root/"data.yaml"

DATA_YAML = prepare_dataset(image_zip_bytes, label_zip_bytes, CLASS_NAMES,
                            "/content/unit07_dataset")
print("設定檔：", DATA_YAML)
DATA_CLASS_NAMES = tuple(CLASS_NAMES)
DATA_SOURCE_SIGNATURE = CLASS_SIGNATURE


## 06 再次核對標註與名稱
這裡顯示的是 **原始標註，不是模型偵測結果**。框上顯示 ID，每張圖下方顯示「ID = 名稱」，中文名稱也可正常顯示。

核對最多 6 張預覽，正確後勾選並按「確認標註，允許訓練」，再執行 07。全部照片可在 04 的各類選單瀏覽。

若名稱相反，回 04 依照片修正欄位，再重跑 05、06。若標註本身混亂，先修正 ZIP 並重新上傳。改過類別後舊的確認會失效，避免拿舊設定訓練。


In [ ]:
REVIEW_SIGNATURE = None

def current_data_key():
    return (dataset_signature(image_zip_bytes,label_zip_bytes),tuple(CLASS_NAMES))

assert CLASS_NAMES and CLASS_SIGNATURE==dataset_signature(image_zip_bytes,label_zip_bytes), "請先完成 04。"
assert DATA_SOURCE_SIGNATURE==CLASS_SIGNATURE and DATA_CLASS_NAMES==tuple(CLASS_NAMES), "類別或 ZIP 已變更，請重新執行 05。"
preview_key=current_data_key()
root=DATA_YAML.parent
print("這是標註預覽，不是 AI 預測。請核對框線 ID 與下方名稱。")
for split in ("train","val"):
    for photo in sorted((root/"images"/split).glob("*"))[:3]:
        rows=read_label_rows((root/"labels"/split/(photo.stem+".txt")).read_text(encoding="utf-8-sig"))
        image=widgets.Image(value=preview_png(photo.read_bytes(),rows),format="png",layout=widgets.Layout(max_width="480px"))
        names=[widgets.Label(value=f"ID {cid} = {CLASS_NAMES[cid]}") for cid in sorted({r[0] for r in rows})]
        display(widgets.VBox([widgets.Label(value=split+" / "+photo.name),image]+names))
print("若名稱與紅框物件不符：回 04 依照片修正名稱，再重跑 05、06。若同一 ID 混合不同物件，請先修正原始標註。")
if globals().get("review_form") is not None: review_form.close()
review_check=widgets.Checkbox(value=False,description="我已確認照片中的物件與 ID 名稱相符",indent=False,layout=widgets.Layout(width="95%"))
review_button=widgets.Button(description="確認標註，允許訓練",button_style="success",layout=widgets.Layout(width="240px"))
review_feedback=widgets.Output()
def reset_review(change):
    global REVIEW_SIGNATURE
    REVIEW_SIGNATURE=None
review_check.observe(reset_review,names="value")
def approve_review(button):
    global REVIEW_SIGNATURE
    REVIEW_SIGNATURE=None
    with review_feedback:
        clear_output()
        if not review_check.value:
            print("請先核對照片並勾選確認。")
        elif current_data_key()!=preview_key or DATA_CLASS_NAMES!=tuple(CLASS_NAMES) or CLASS_SIGNATURE!=DATA_SOURCE_SIGNATURE:
            print("資料或類別已變更，請重跑 04、05、06。")
        else:
            REVIEW_SIGNATURE=preview_key
            print("標註確認完成，可以執行 07 訓練。")
review_button.on_click(approve_review)
review_form=widgets.VBox([review_check,review_button,review_feedback]);display(review_form)

## 07 開始訓練
確認 06 正確後再執行。預設最多 50 回合；15 回合未改善可能提早停止，這不是失敗。不要再次點擊正在執行的儲存格。

若顯示 CUDA out of memory，等本格停止後，把 BATCH 改為 8，仍不足再改為 4，重新執行本格。每次訓練使用不同時間資料夾，避免拿到舊模型。

In [ ]:
assert CLASS_NAMES and CLASS_SIGNATURE == dataset_signature(image_zip_bytes, label_zip_bytes), "請先完成 04。"
assert DATA_CLASS_NAMES == tuple(CLASS_NAMES) and DATA_SOURCE_SIGNATURE == CLASS_SIGNATURE, "請重新執行 05。"
assert REVIEW_SIGNATURE == current_data_key(), "請先在 06 核對照片並按確認標註，才能訓練。"
from ultralytics import YOLO
from datetime import datetime
BATCH = 16
RUN_NAME = "unit07_" + datetime.now().strftime("%Y%m%d_%H%M%S")
model = YOLO("yolo26n.pt")
model.train(data=str(DATA_YAML), epochs=50, imgsz=640,
            batch=BATCH, device=0, patience=15,
            project="/content/runs", name=RUN_NAME, workers=2,
            seed=42, plots=True)
RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR/"weights"/"best.pt"
assert BEST_PT.exists(), "找不到 best.pt；請先確認訓練成功。"
print("訓練完成，最佳模型：", BEST_PT)
from IPython.display import display, Image as DisplayImage
for name in ("results.png", "val_batch0_pred.jpg"):
    if (RUN_DIR/name).exists():
        display(DisplayImage(filename=str(RUN_DIR/name), width=900))

## 08 匯出 ONNX 並檢查
不改參數。沿用固定 640 × 640、batch 1、FP32、opset 18，保留類別名稱。匯出不會重新訓練。

本格會檢查 ONNX 結構、輸入輸出及 names，並以空白張量試跑一次，確認可執行；這不是辨識準確率測試。成功才繼續下載。

In [ ]:
import ast, onnx, onnxruntime as ort, numpy as np
trained = YOLO(str(BEST_PT))
ONNX_PATH = Path(trained.export(format="onnx", imgsz=640,
                               batch=1, dynamic=False, half=False,
                               simplify=True, opset=18, device="cpu"))
onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
metadata = {p.key:p.value for p in onnx_model.metadata_props}
assert "names" in metadata, "模型缺少 names，請停止並告知老師。"
model_names = ast.literal_eval(metadata["names"])
expected_names = dict(enumerate(CLASS_NAMES))
assert model_names == expected_names, "模型類別與 04 不一致，請確認使用本次 best.pt。"
check_session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
inp = check_session.get_inputs()[0]
assert inp.shape == [1,3,640,640] and inp.type == "tensor(float)", "模型輸入不符合 STEP4。"
outputs = check_session.run(None, {inp.name:np.zeros((1,3,640,640), dtype=np.float32)})
shape = outputs[0].shape
assert len(outputs) == 1 and shape == (1,4+len(CLASS_NAMES),8400), f"輸出 {shape} 與本教材預期不同，請停止並告知老師。"
print("ONNX 檢查通過")
print("輸入：", inp.shape, inp.type)
print("輸出：", shape)
print("類別：", model_names)
print("檔案：", ONNX_PATH)
print("大小 MB：", round(ONNX_PATH.stat().st_size/1024**2, 2))
del check_session, outputs, onnx_model

## 09 下載 best.onnx
執行後等待下載。請確認電腦下載資料夾真的有 best.onnx；不要把 best.pt 改副檔名當成 ONNX。若瀏覽器擋住，允許下載後重跑本格。

In [ ]:
assert ONNX_PATH.exists(), "請先成功執行 08。"
files.download(str(ONNX_PATH))

## 10 下載備份並結束
備份包含 best.pt、best.onnx、data.yaml、classes.txt、訓練紀錄與版本資訊，不含原始照片。原始照片與標註 ZIP 請自行保留。確認下載完成後再中斷執行階段。

In [ ]:
import json
BACKUP = Path("/content/unit07_model_backup.zip")
record = {"ultralytics":ultralytics.__version__, "torch":torch.__version__,
          "classes":CLASS_NAMES, "imgsz":640, "epochs_max":50,
          "batch":BATCH, "patience":15, "run_dir":str(RUN_DIR),
          "onnx_opset":18, "onnx":onnx.__version__, "onnxruntime":ort.__version__}
with zipfile.ZipFile(BACKUP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (BEST_PT, ONNX_PATH, DATA_YAML, DATA_YAML.parent/"classes.txt"):
        archive.write(path, path.name)
    for name in ("results.csv", "results.png", "args.yaml", "val_batch0_pred.jpg"):
        path = RUN_DIR/name
        if path.exists(): archive.write(path, name)
    archive.writestr("training_record.json", json.dumps(record, ensure_ascii=False, indent=2))
files.download(str(BACKUP))
print("請確認 best.onnx 與 unit07_model_backup.zip 都已下載至自己的電腦。")

## 下一步 回到 UNIT07 STEP4
開啟教師提供的 HTTPS 網頁，載入 best.onnx，確認自動讀取的類別名稱與數量，再啟動攝影機。

本次名稱在 04 設定後會寫入訓練模型；STEP4 可直接讀取，不需再逐一輸入。

參考資料：
- https://docs.ultralytics.com/modes/train/
- https://docs.ultralytics.com/modes/export/
- https://docs.ultralytics.com/datasets/detect/
- https://research.google.com/colaboratory/intl/zh-TW/faq.html